# Twin Experiment Framework: Inverse Crime Avoidance Testing

This notebook tests the twin experiment framework with physics perturbations to avoid the "inverse crime" - using the exact same model for both truth generation and data assimilation.

## Experimental Design Matrix

| Experiment | Bathymetry | Friction | Purpose |
|------------|------------|----------|----------|
| **Baseline** | Same | Same | Sanity check (inverse crime) |
| **A** | Perturbed | Same | Bathymetry error only |
| **B** | Same | Perturbed | Friction error only |
| **C** | Perturbed | Perturbed | Combined (realistic scenario) |

Each experiment is run with both:
- **4D-Var**: Standard variational data assimilation
- **DC-WME**: Data-Consistent Weighted Mean Error (more robust to model error)

## Expected Results

- **Baseline**: Both methods should perform well (artificially favorable due to inverse crime)
- **Experiments A, B, C**: Performance should degrade as model error increases
- **DC-WME** should be more robust to model error than standard 4D-Var

## Setup and Imports

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import asdict

# Add project root to path
sys.path.insert(0, os.path.dirname(os.getcwd()))

from experiments.twin_experiment import TwinExperiment, TwinExperimentConfig
from swe4dvar.forward.problems import TidalProblem
from swe4dvar.forward.solvers import get_solver

print("Imports successful!")

## Problem Configuration

We use a small TidalProblem for quick testing:
- 5x3 element mesh
- 1 hour timestep
- 12 timesteps (12 hours total)
- Flat bathymetry at 10m depth
- Constant friction coefficient (TAU = 0.02)

In [ ]:
# Problem parameters (small for quick testing)
NX = 5
NY = 3
DT = 3600  # 1 hour
NT = 12    # 12 timesteps

# Common experiment settings
COMMON_CONFIG = {
    "obs_fraction": 0.5,
    "obs_frequency": 1,
    "obs_noise_level": 0.01,
    "background_error_std": 0.1,
    "max_iterations": 30,
    "verbose": False,  # Reduce output for notebook
    "obs_seed": 42,
    "background_seed": 123,
    "perturbation_seed": 456,
}

# Physics perturbation settings
BATHYMETRY_NOISE_STD = 0.5  # 0.5m additive noise (5% of 10m depth)
BATHYMETRY_CORRELATION_LENGTH = 500.0  # meters
FRICTION_SCALE_FACTOR = 1.15  # 15% increase in friction

print(f"Problem: {NX}x{NY} elements, dt={DT}s, nt={NT}")
print(f"Bathymetry perturbation: {BATHYMETRY_NOISE_STD}m additive noise")
print(f"Friction perturbation: {FRICTION_SCALE_FACTOR}x scale factor")

## Helper Functions

In [ ]:
def create_problem_and_solver():
    """Create a fresh problem and solver instance."""
    problem = TidalProblem(nx=NX, ny=NY, dt=DT, nt=NT)
    # Set verbose=False to suppress solver output (Newton iterations, timestep progress)
    solver = get_solver("SUPG")(problem, theta=0.5, p_degree=[1, 1], verbose=False)
    return problem, solver


def run_experiment(name: str, config: TwinExperimentConfig):
    """Run a single twin experiment and return results."""
    print(f"\n{'='*60}")
    print(f"Running: {name}")
    print(f"{'='*60}")
    
    problem, solver = create_problem_and_solver()
    experiment = TwinExperiment(problem, solver, config)
    results = experiment.run()
    
    print(f"\nResults for {name}:")
    print(f"  Background error: {results.background_error:.6f}")
    print(f"  Analysis error:   {results.analysis_error:.6f}")
    print(f"  Error reduction:  {results.error_reduction:.1f}%")
    print(f"  Converged:        {results.converged}")
    print(f"  Iterations:       {results.num_iterations}")
    
    return results


def plot_comparison(results_dict: dict, title: str):
    """Plot comparison of results across experiments."""
    experiments = list(results_dict.keys())
    
    # Extract metrics
    bg_errors = [results_dict[exp].background_error for exp in experiments]
    an_errors = [results_dict[exp].analysis_error for exp in experiments]
    reductions = [results_dict[exp].error_reduction for exp in experiments]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Plot 1: Background vs Analysis Error
    x = np.arange(len(experiments))
    width = 0.35
    
    axes[0].bar(x - width/2, bg_errors, width, label='Background Error', alpha=0.8)
    axes[0].bar(x + width/2, an_errors, width, label='Analysis Error', alpha=0.8)
    axes[0].set_xlabel('Experiment')
    axes[0].set_ylabel('RMS Error')
    axes[0].set_title(f'{title}: Errors')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(experiments, rotation=45, ha='right')
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # Plot 2: Error Reduction
    colors = ['green' if r > 0 else 'red' for r in reductions]
    axes[1].bar(x, reductions, color=colors, alpha=0.8)
    axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    axes[1].set_xlabel('Experiment')
    axes[1].set_ylabel('Error Reduction (%)')
    axes[1].set_title(f'{title}: Error Reduction')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(experiments, rotation=45, ha='right')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()


# Storage for all results
all_results = {
    "4dvar": {},
    "dcwme": {},
}

---

# Experiment Baseline: Inverse Crime (No Perturbation)

This is the **baseline** experiment where the same model is used for both truth generation and data assimilation. This represents the "inverse crime" scenario.

**Configuration:**
- Bathymetry: Same as truth (no perturbation)
- Friction: Same as truth (no perturbation)

**Expected behavior:**
- Both 4D-Var and DC-WME should perform very well
- High error reduction (likely > 50%)
- This provides an upper bound on performance that may be unrealistic in practice

In [ ]:
# Baseline configuration - no physics perturbation (inverse crime)
config_baseline = TwinExperimentConfig(
    **COMMON_CONFIG,
    method="4dvar",
    perturb_bathymetry=False,
    perturb_friction=False,
)

all_results["4dvar"]["Baseline"] = run_experiment("Baseline (4D-Var)", config_baseline)

In [ ]:
# Baseline with DC-WME
config_baseline_dcwme = TwinExperimentConfig(
    **COMMON_CONFIG,
    method="dcwme",
    perturb_bathymetry=False,
    perturb_friction=False,
)

all_results["dcwme"]["Baseline"] = run_experiment("Baseline (DC-WME)", config_baseline_dcwme)

---

# Experiment A: Bathymetry Perturbation Only

This experiment introduces **bathymetry error** while keeping friction the same.

**Configuration:**
- Bathymetry: Perturbed with additive smooth noise (0.5m std, 500m correlation length)
- Friction: Same as truth

**Physical interpretation:**
- Represents uncertainty in bed elevation measurements
- Additive noise is appropriate for flat bathymetry (TidalProblem has 10m constant depth)
- Spatially correlated noise is more realistic than white noise

**Expected behavior:**
- Performance should degrade compared to baseline
- DC-WME may show better robustness than 4D-Var

In [ ]:
# Experiment A: Bathymetry perturbation only (4D-Var)
config_A = TwinExperimentConfig(
    **COMMON_CONFIG,
    method="4dvar",
    perturb_bathymetry=True,
    bathymetry_noise_std=BATHYMETRY_NOISE_STD,
    bathymetry_noise_type="additive",
    bathymetry_correlation_length=BATHYMETRY_CORRELATION_LENGTH,
    perturb_friction=False,
)

all_results["4dvar"]["A: Bathy"] = run_experiment("Exp A: Bathymetry Only (4D-Var)", config_A)

In [ ]:
# Experiment A: Bathymetry perturbation only (DC-WME)
config_A_dcwme = TwinExperimentConfig(
    **COMMON_CONFIG,
    method="dcwme",
    perturb_bathymetry=True,
    bathymetry_noise_std=BATHYMETRY_NOISE_STD,
    bathymetry_noise_type="additive",
    bathymetry_correlation_length=BATHYMETRY_CORRELATION_LENGTH,
    perturb_friction=False,
)

all_results["dcwme"]["A: Bathy"] = run_experiment("Exp A: Bathymetry Only (DC-WME)", config_A_dcwme)

---

# Experiment B: Friction Perturbation Only

This experiment introduces **friction error** while keeping bathymetry the same.

**Configuration:**
- Bathymetry: Same as truth
- Friction: Scaled by factor of 1.15 (15% increase)

**Physical interpretation:**
- Represents uncertainty in bottom friction/roughness
- Uniform scaling is simple but effective for testing
- Increased friction means slower flow velocities in the DA model

**Expected behavior:**
- Performance should degrade compared to baseline
- Effect may be different from bathymetry perturbation

In [ ]:
# Experiment B: Friction perturbation only (4D-Var)
config_B = TwinExperimentConfig(
    **COMMON_CONFIG,
    method="4dvar",
    perturb_bathymetry=False,
    perturb_friction=True,
    friction_scale_factor=FRICTION_SCALE_FACTOR,
)

all_results["4dvar"]["B: Friction"] = run_experiment("Exp B: Friction Only (4D-Var)", config_B)

In [ ]:
# Experiment B: Friction perturbation only (DC-WME)
config_B_dcwme = TwinExperimentConfig(
    **COMMON_CONFIG,
    method="dcwme",
    perturb_bathymetry=False,
    perturb_friction=True,
    friction_scale_factor=FRICTION_SCALE_FACTOR,
)

all_results["dcwme"]["B: Friction"] = run_experiment("Exp B: Friction Only (DC-WME)", config_B_dcwme)

---

# Experiment C: Combined Perturbations (Realistic)

This experiment combines **both bathymetry and friction errors** for the most realistic scenario.

**Configuration:**
- Bathymetry: Perturbed with additive smooth noise (0.5m std)
- Friction: Scaled by factor of 1.15 (15% increase)

**Physical interpretation:**
- Represents a realistic operational scenario
- Both depth and roughness are uncertain in real applications
- This is the most challenging test for the DA system

**Expected behavior:**
- This should show the largest performance degradation
- Provides the most realistic assessment of DA performance
- DC-WME's robustness advantage should be most apparent here

In [ ]:
# Experiment C: Combined perturbations (4D-Var)
config_C = TwinExperimentConfig(
    **COMMON_CONFIG,
    method="4dvar",
    perturb_bathymetry=True,
    bathymetry_noise_std=BATHYMETRY_NOISE_STD,
    bathymetry_noise_type="additive",
    bathymetry_correlation_length=BATHYMETRY_CORRELATION_LENGTH,
    perturb_friction=True,
    friction_scale_factor=FRICTION_SCALE_FACTOR,
)

all_results["4dvar"]["C: Combined"] = run_experiment("Exp C: Combined (4D-Var)", config_C)

In [ ]:
# Experiment C: Combined perturbations (DC-WME)
config_C_dcwme = TwinExperimentConfig(
    **COMMON_CONFIG,
    method="dcwme",
    perturb_bathymetry=True,
    bathymetry_noise_std=BATHYMETRY_NOISE_STD,
    bathymetry_noise_type="additive",
    bathymetry_correlation_length=BATHYMETRY_CORRELATION_LENGTH,
    perturb_friction=True,
    friction_scale_factor=FRICTION_SCALE_FACTOR,
)

all_results["dcwme"]["C: Combined"] = run_experiment("Exp C: Combined (DC-WME)", config_C_dcwme)

---

# Results Summary

Now we compare the results across all experiments and methods.

In [ ]:
# Print summary table
print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)
print(f"{'Experiment':<20} {'Method':<10} {'Bg Error':<12} {'An Error':<12} {'Reduction':<12} {'Converged'}")
print("-"*80)

for method in ["4dvar", "dcwme"]:
    for exp_name, results in all_results[method].items():
        print(f"{exp_name:<20} {method.upper():<10} {results.background_error:<12.6f} "
              f"{results.analysis_error:<12.6f} {results.error_reduction:<12.1f}% {results.converged}")

print("="*80)

In [ ]:
# Plot 4D-Var results
plot_comparison(all_results["4dvar"], "4D-Var")

In [ ]:
# Plot DC-WME results
plot_comparison(all_results["dcwme"], "DC-WME")

In [ ]:
# Compare 4D-Var vs DC-WME for each experiment
experiments = list(all_results["4dvar"].keys())

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(experiments))
width = 0.35

reductions_4dvar = [all_results["4dvar"][exp].error_reduction for exp in experiments]
reductions_dcwme = [all_results["dcwme"][exp].error_reduction for exp in experiments]

bars1 = ax.bar(x - width/2, reductions_4dvar, width, label='4D-Var', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, reductions_dcwme, width, label='DC-WME', color='darkorange', alpha=0.8)

ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_xlabel('Experiment')
ax.set_ylabel('Error Reduction (%)')
ax.set_title('Error Reduction: 4D-Var vs DC-WME')
ax.set_xticks(x)
ax.set_xticklabels(experiments)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, val in zip(bars1, reductions_4dvar):
    ax.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width()/2, val),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, reductions_dcwme):
    ax.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width()/2, val),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## Convergence History Comparison

In [ ]:
# Plot convergence history for all experiments
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 4D-Var convergence
for exp_name, results in all_results["4dvar"].items():
    if results.cost_history:
        normalized_cost = np.array(results.cost_history) / results.cost_history[0]
        axes[0].semilogy(normalized_cost, label=exp_name, linewidth=2)

axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Normalized Cost (J/J₀)')
axes[0].set_title('4D-Var Convergence History')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# DC-WME convergence
for exp_name, results in all_results["dcwme"].items():
    if results.cost_history:
        normalized_cost = np.array(results.cost_history) / results.cost_history[0]
        axes[1].semilogy(normalized_cost, label=exp_name, linewidth=2)

axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Normalized Cost (J/J₀)')
axes[1].set_title('DC-WME Convergence History')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Analysis

### Key Observations

1. **Baseline (Inverse Crime)**: 
   - Both methods achieve high error reduction
   - This represents an unrealistic upper bound on performance

2. **Experiment A (Bathymetry)**: 
   - Performance degrades due to depth uncertainty
   - Compare how 4D-Var and DC-WME handle this error source

3. **Experiment B (Friction)**:
   - Performance degrades due to friction uncertainty
   - Friction affects velocity damping throughout the domain

4. **Experiment C (Combined)**:
   - Most realistic scenario with multiple error sources
   - Shows the practical limits of each DA method

### Conclusions

- The "inverse crime" baseline gives artificially optimistic results
- Physics perturbation provides more realistic performance estimates
- Compare 4D-Var vs DC-WME robustness to model error

In [ ]:
# Calculate and display statistics
print("\n" + "="*60)
print("ANALYSIS: Impact of Model Error on DA Performance")
print("="*60)

for method in ["4dvar", "dcwme"]:
    print(f"\n{method.upper()}:")
    baseline_reduction = all_results[method]["Baseline"].error_reduction
    
    for exp_name, results in all_results[method].items():
        if exp_name != "Baseline":
            degradation = baseline_reduction - results.error_reduction
            print(f"  {exp_name}: {degradation:.1f}% degradation from baseline")

In [ ]:
# Compare method robustness
print("\n" + "="*60)
print("ROBUSTNESS COMPARISON: DC-WME vs 4D-Var")
print("="*60)

for exp_name in experiments:
    reduction_4dvar = all_results["4dvar"][exp_name].error_reduction
    reduction_dcwme = all_results["dcwme"][exp_name].error_reduction
    diff = reduction_dcwme - reduction_4dvar
    
    if diff > 0:
        better = "DC-WME"
    elif diff < 0:
        better = "4D-Var"
    else:
        better = "Tie"
    
    print(f"{exp_name}: {better} better by {abs(diff):.1f}% error reduction")